In [14]:
import pandas as pd
import os
from pathlib import Path
import shutil
import re
import unicodedata

Crear directorio `csv_cleaned`

In [15]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

Cargar archivos csv

In [16]:
df = pd.read_csv(root / "csv_enunciados/ejercicios_enunciados.csv", encoding="utf-8")

In [17]:
def clean_dataframe(df):
    df_sin_metadatos = df[df.aviso == "sin metadatos"]
    df = df[df.aviso.isna()]

    # modify ejercicio column
    df.ejercicio = df.ejercicio.apply(lambda x : x.replace("Opción", "opción")
                                      .replace(".", ",") if pd.notna(x) else x)

    # remove gravity from physics statements
    patron = r"g\s*=\s*9[,.]8\s*m\s*[·.]?\s*s\s*-\s*2\s*"
    mask = df.asignatura == "Física"
    df.loc[mask, "enunciado"] = df.loc[mask, "enunciado"].str.replace(patron, "", regex=True)
    
    # normalize text
    df.enunciado = df.enunciado.apply(lambda texto : unicodedata.normalize("NFKC", texto)
                                      if pd.notna(texto) else texto)

    # trim multiple spaces
    df.enunciado = df.enunciado.apply(lambda texto : re.sub(r"[ \t]+", " ", texto)
                                      .strip() if pd.notna(texto) else texto)

    # eliminar , al final del ejercicio
    df.ejercicio = df.ejercicio.str.strip(",")

    # create folder for clean csv
    output_dir = Path(root / "csv_cleaned")
    output_dir.mkdir(exist_ok=True)

    return df

Arreglar ejercicios de Ponencia

In [18]:
def fix_ponencia(row):
    if "PONENCIA" in row["info_ejercicio"]:
        row["convocatoria"] = "Ponencia"
        pos_ejercicio = row["info_ejercicio"].find("EJERCICIO")
        row["ejercicio"] = row["info_ejercicio"][pos_ejercicio:].title()
    return row

In [19]:
df = df.apply(fix_ponencia, axis = 1)

In [20]:
df[df.ejercicio.isna()]

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
2556,Mates II,Mates II,2017,NaN,NaN,Matrices Y Determinantes,MATEMÁTICAS II. 2017. RESERR VE A S 3 .O E JLE...,Sea A una matriz 3x3 tal que det(2A)8.\na) ¿C...,source_pdf_files/Mates II/matrices_determinant...,enunciados_ejercicios/Mates II/matrices_determ...,NaN


In [21]:
df.loc[2556, "info_ejercicio"]

'MATEMÁTICAS II. 2017. RESERR VE A S 3 .O E JLE RUC CIC II O Ó 3 .N O PCIÓN B.'

In [22]:
df.loc[2556, ["convocatoria", "ejercicio"]] = ["Reserva 3", "Ejercicio 3, opción B"]

In [23]:
df.sample()

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
529,Mates CCSS,Mates CCSS,2002,Junio,Ejercicio 1 Opción B,Programación Lineal,SOCIALES II. 2002 JUNIO. EJERCICIO 1 OPCIÓN B,Sea el sistema de inecuaciones siguiente: x y...,source_pdf_files/Mates CCSS/programación_line...,enunciados_ejercicios/Mates CCSS/programación...,NaN


In [24]:
df = clean_dataframe(df)

Tras la limpieza completa, creo el ID

In [25]:
def create_id(row):
    partes = [row["asignatura"], row["tema"], row["año"], row["convocatoria"]]
    if pd.notna(row["ejercicio"]):
        partes.append(row["ejercicio"])
    id_string = "_".join(str(p) for p in partes).lower()

    # quitar acentos: descomponer y eliminar los caracteres combinantes
    id_string = unicodedata.normalize("NFKD", id_string)
    id_string = "".join(c for c in id_string if not unicodedata.combining(c))

    # dejar solo letras, números y _ (quita comas, puntos, espacios...)
    id_string = re.sub(r"[^a-z0-9]+", "_", id_string).strip("_")
    return id_string

In [26]:
# create id
df["id"] = df.apply(lambda row: create_id(row), axis = 1)
    
# rearrange columns
df = df[["id"] + list(df.columns[:-2])]

Exportar el dataframe limpio

In [27]:
df = (df.sort_values(["asignatura", "tema", "año", "convocatoria", "ejercicio"],
               ascending = [True, True, False, True, True])
               .reset_index(drop = True)
)

In [28]:
df.sample(5)

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
4419,quimica_reacciones_redox_2000_reserva_3_ejerci...,Química,Química,2000,Reserva 3,"Ejercicio 3, opción A",Reacciones Redox,QUÍMICA. 2000. RESERVA 3. EJERCICIO 3. OPCIÓN A,a) ¿Reaccionará una disolución acuosa de ácido...,source_pdf_files/Química/redox/2000 - Reaccio...,enunciados_ejercicios/Química/redox/png/png_Q...
1312,mates_ccss_probabilidad_2018_reserva_4_ejercic...,Mates CCSS,Mates CCSS,2018,Reserva 4,Ejercicio 3 opción A,Probabilidad,SOCIALES II. 2018 RESERVA 4. EJERCICIO 3 OPCIÓN A,Se ha realizado un referéndum en el que se ha ...,source_pdf_files/Mates CCSS/probabilidad/2018 ...,enunciados_ejercicios/Mates CCSS/probabilidad/...
1378,mates_ccss_probabilidad_2012_reserva_1_ejercic...,Mates CCSS,Mates CCSS,2012,Reserva 1,"Ejercicio 3, opción A",Probabilidad,S OCIALES II. 2012. RESERVA 1. EJERCICIO 3. OP...,E n un congreso de 200 jóvenes profesionales s...,source_pdf_files/Mates CCSS/probabilidad/2012 ...,enunciados_ejercicios/Mates CCSS/probabilidad/...
1458,mates_ccss_probabilidad_2006_septiembre_ejerci...,Mates CCSS,Mates CCSS,2006,Septiembre,"Ejercicio 3, Parte I, opción A",Probabilidad,SOCIALES II. 2006. SEPTIEMBRE. EJERCICIO 3. PA...,Laura tiene un dado con tres caras pintadas de...,source_pdf_files/Mates CCSS/probabilidad/2006 ...,enunciados_ejercicios/Mates CCSS/probabilidad/...
718,mates_ccss_funciones_ccss_2011_reserva_2_ejerc...,Mates CCSS,Mates CCSS,2011,Reserva 2,"Ejercicio 2, opción B",Funciones CCSS,SOCIALES II. 2011. RESERVA 2. EJERCICIO 2. OPC..., 2\n si x0\n\n x2\nSe considera la funci...,source_pdf_files/Mates CCSS/funciones_ccss/201...,enunciados_ejercicios/Mates CCSS/funciones_ccs...


In [29]:
df.isna().sum()

id                    0
asignatura            0
asignatura_carpeta    0
año                   0
convocatoria          0
ejercicio             0
tema                  0
info_ejercicio        0
enunciado             4
archivo               0
imagen                0
dtype: int64

In [30]:
df[df.enunciado.isna()]

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
442,fisica_ondas_2016_reserva_3_ejercicio_4_opcion_a,Física,Física,2016,Reserva 3,"Ejercicio 4, opción A",Ondas,FISICA. 2016. RESERVA 3. EJERCICIO 4. OPCIÓN A,NaN,source_pdf_files/Física/movimiento_ondulatori...,enunciados_ejercicios/Física/movimiento_ondul...
4204,quimica_reacciones_redox_2026_junio_ejercicio_5,Química,Química,2026,Junio,Ejercicio 5,Reacciones Redox,QUÍMICA. 2026. JUNIO. EJERCICIO 5,NaN,source_pdf_files/Química/redox/2026 - Reaccio...,enunciados_ejercicios/Química/redox/png/png_Q...
4652,quimica_termoquimica_2025_julio_ejercicio_5,Química,Química,2025,Julio,Ejercicio 5,Termoquímica,QUÍMICA.2025. JULIO. EJERCICIO 5,NaN,source_pdf_files/Química/termoquímica/2025 -...,enunciados_ejercicios/Química/termoquímica/p...
4655,quimica_termoquimica_2025_reserva_2_ejercicio_5,Química,Química,2025,Reserva 2,Ejercicio 5,Termoquímica,QUÍMICA.2025. RESERVA 2. EJERCICIO 5,NaN,source_pdf_files/Química/termoquímica/2025 -...,enunciados_ejercicios/Química/termoquímica/p...


In [31]:
df.asignatura.unique()

<StringArray>
['Física', 'Mates CCSS', 'Mates II', 'Química']
Length: 4, dtype: str

In [32]:
df.loc[:, ["asignatura", "tema"]].drop_duplicates()

,asignatura,tema
0,Física,Campo eléctrico y magnético
121,Física,Campo gravitatorio
244,Física,Física cuántica y nuclear
362,Física,Ondas
449,Física,Óptica geométrica
486,Mates CCSS,Contraste De Hipótesis
545,Mates CCSS,Distribución Binomial
546,Mates CCSS,Distribución Binomial
550,Mates CCSS,Funciones CCSS
806,Mates CCSS,Inferencia Estadística


In [33]:
df.tema = df.tema.str.strip()

In [34]:
df.loc[df.tema == "Matrices Y Determinantes", "tema"] = "Matrices y determinantes"

In [35]:
df[df.asignatura == "Mates CCSS"].tema.unique()

<StringArray>
[        'Contraste De Hipótesis',          'Distribución Binomial',
                  'Funciones CCSS',         'Inferencia Estadística',
        'Matrices y determinantes',                    'Probabilidad',
            'Programación Lineal', 'Sistemas de Ecuaciones Lineales']
Length: 8, dtype: str

In [36]:
df.convocatoria.unique()

<StringArray>
[             'Julio',              'Junio',          'Reserva 1',
          'Reserva 2',          'Reserva 3',          'Reserva 4',
         'Septiembre', 'Reserva1 Ejercicio',           'Ponencia',
             'Modelo']
Length: 10, dtype: str

In [37]:
df[df.convocatoria == "Reserva1 Ejercicio"]

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
195,fisica_campo_gravitatorio_2020_reserva1_ejerci...,Física,Física,2020,Reserva1 Ejercicio,1,Campo gravitatorio,FISICA. 2020. RESERVA1. EJERCICIO 1,a) Considere dos partículas de igual masa sepa...,source_pdf_files/Física/campo_gravitatorio/20...,enunciados_ejercicios/Física/campo_gravitator...


In [38]:
df.loc[223, ["convocatoria", "ejercicio"]] = ("Reserva 1", "Ejercicio 1")

In [39]:
df.to_csv(root/"csv_enunciados/enunciados_ejercicios_clean.csv", index = False, encoding="utf-8")

In [40]:
df = pd.read_csv(root / "csv_enunciados/enunciados_ejercicios_clean.csv", encoding="utf-8")

In [41]:
df.sample(3)

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
4039,quimica_formulacion_2013_reserva_1_ejercicio_1...,Química,Química,2013,Reserva 1,"Ejercicio 1, opción B",Formulación,QUÍMICA. 2013. RESERVA 1. EJERCICIO 1. OPCIÓN B,Formule o nombre los siguientes compuestos: a)...,source_pdf_files/Química/formulación/2013 - ...,enunciados_ejercicios/Química/formulación/pn...
833,mates_ccss_inferencia_estadistica_2024_reserva...,Mates CCSS,Mates CCSS,2024,Reserva 4,Ejercicio D8,Inferencia Estadística,SOCIALES II. 2024 RESERVA 4. EJERCICIO D8,Una empresa farmacéutica desea revisar la efec...,source_pdf_files/Mates CCSS/inferencia_estadí...,enunciados_ejercicios/Mates CCSS/inferencia_es...
4627,quimica_solubilidad_2015_junio_ejercicio_3_opc...,Química,Química,2015,Junio,"Ejercicio 3, opción A",Solubilidad,QUÍMICA. 2015. JUNIO. EJERCICIO 3. OPCIÓN A,"Dada una disolución saturada de Mg(OH) , cuyo ...",source_pdf_files/Química/solubilidad/2015 - S...,enunciados_ejercicios/Química/solubilidad/png...


In [42]:
df.loc[0, :].imagen

'enunciados_ejercicios/Física/campo_eléctrico_magnético/png/png_Física_2025 - Campo eléctrico y magnético/Física_campo_eléctrico_magnético_2025_julio_ejercicio_a.png'